# Lightweight Baseline (Reference Starter)


## Lightweight constraints (Final Presentation)

| Requirement | This baseline |
|---|---|
| CPU-only (no GPU) | `gpu=False` everywhere |
| Small model / memory | Product head ≈ **few MB** (sklearn TF-IDF + LogisticRegression) |
| Fast product inference | **<1 ms** per image after OCR |
| Deployable pattern | Rules + tiny classifier on OCR text |

> EasyOCR (~200 MB weights) is used here as a **convenient OCR demo**. For production, swap in a lighter OCR (PaddleOCR mobile, Tesseract, custom tiny CNN) and keep the same product head.

## Scoring Formula

$$\text{Score} = 0.6 \times F1_{\text{product}} + 0.4 \times (1 - \text{CER})$$

## Pipeline (what to copy)

1. **OCR** — pluggable backend (EasyOCR in this notebook)  
2. **Product** — `BRAND_RULES` (zero MB) + trainable head from `train_labels.csv`  
3. **Evaluate** — composite metric (Cell 7)

## Required Files (Kaggle Input)

| File | Description |
|---|---|
| `test.csv` | Image IDs |
| `test_images/images/` | Test JPEGs (`img_XXXX.jpg`) — Kaggle competition mount |
| `sample_submission.csv` | Format template |
| `train_labels.csv` | Draft labels for the lightweight product head |

> **Kaggle path:** `/kaggle/input/competitions/the-2nd-ura-hackathon/`  
> Add Input → **Competition Data** (not a manual Dataset upload).  
> Local dev may use `test/test_images.zip` instead of `test_images/`.

## Notebook Outline

1. Install dependencies  
2. Load data  
3. Brand rules (zero-cost baseline)  
4. Train lightweight product model (~seconds, few MB)  
5. OCR engine  
6. Main loop → `submission.csv`  
7. Validate export  
8. Local composite score


## Cell 1 — Install Dependencies

Run once. Most packages are preinstalled on Kaggle; this cell ensures EasyOCR is available.

In [1]:
!pip install easyocr tqdm scikit-learn -q
print("Install done")


Install done


## Cell 2 — Load Data

Read `test.csv`, auto-detect the competition input path, and use `test_images/` (or extract a zip locally).

If `train_labels.csv` is attached to the same Kaggle dataset, it is loaded for optional supervised learning (draft v1 weak labels).

No Internet required — images are read from the competition dataset (see Dataset Description).

In [2]:
import re, time, zipfile
import pandas as pd
from pathlib import Path

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

try:
    from PIL import Image, ImageEnhance, ImageFilter
    HAS_PIL = True
except ImportError:
    HAS_PIL = False


# Kaggle competition mount: test_images/images/img_XXXX.jpg
IMAGE_DIR_CANDIDATES = (
    "test_images/images",
    "test_images",
    "images",
    "test/images",
    "test/test_images/images",
    "test/test_images",
)
IMAGE_ZIP_NAMES = (
    "test_images.zip",
    "images.zip",
    "test/test_images.zip",
    "test/images.zip",
)


def _input_roots() -> list[Path]:
    """Candidate dataset roots: Kaggle competition mount, datasets, local copies."""
    roots: list[Path] = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        comp_root = kaggle_input / "competitions"
        if comp_root.exists():
            roots.extend(sorted(comp_root.iterdir()))
        roots.extend(sorted(kaggle_input.iterdir()))
    for local in [Path("public_dataset"), Path("smce_dataset/test"), Path(".")]:
        if local.exists():
            roots.append(local.resolve())
    # de-dupe while preserving order
    seen: set[Path] = set()
    out: list[Path] = []
    for r in roots:
        if r not in seen:
            seen.add(r)
            out.append(r)
    return out


def resolve_input_dir() -> Path:
    """Auto-detect dataset root containing test.csv (Kaggle competition or uploaded Data)."""
    for root in _input_roots():
        if (root / "test.csv").exists():
            return root
    # nested test.csv (rare uploads)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for test_csv in sorted(kaggle_input.rglob("test.csv")):
            return test_csv.parent
    hint = (
        "Dataset not found. On Kaggle: Add Input → Competition Data → "
        "The 2nd URA Hackathon (expect test.csv + test_images/ under "
        "/kaggle/input/competitions/the-2nd-ura-hackathon/)."
    )
    if kaggle_input.exists():
        listing = sorted(str(p) for p in kaggle_input.rglob("*") if p.is_file())[:20]
        hint += f"\n/kaggle/input files (first 20): {listing}"
    raise FileNotFoundError(hint)


def _find_images_dir(input_dir: Path) -> Path | None:
    for rel in IMAGE_DIR_CANDIDATES:
        images_dir = input_dir / rel
        if images_dir.is_dir() and any(images_dir.glob("*.jpg")):
            return images_dir
    return None


def _find_images_zip(input_dir: Path) -> Path | None:
    for rel in IMAGE_ZIP_NAMES:
        zip_path = input_dir / rel
        if zip_path.is_file():
            return zip_path
    for zip_path in sorted(input_dir.rglob("*.zip")):
        name = zip_path.name.lower()
        if "train" in name and "test" not in name:
            continue
        if name in ("test_images.zip", "images.zip") or name.endswith("images.zip"):
            return zip_path
    return None


def setup_images_dir(input_dir: Path, work_dir: Path) -> Path:
    """Use test_images/ or images/ from input; otherwise extract a zip once to working/."""
    images_dir = _find_images_dir(input_dir)
    if images_dir is not None:
        return images_dir

    zip_path = _find_images_zip(input_dir)
    if zip_path is None:
        raise FileNotFoundError(
            f"No test images under {input_dir}. Expected test_images/, images/, "
            f"or one of {IMAGE_ZIP_NAMES}."
        )

    extract_root = work_dir / "dataset_images"
    images_dir = extract_root / "images"
    if not any(images_dir.glob("*.jpg")):
        extract_root.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {zip_path} ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_root)
        # zip may unpack as images/, test_images/, or test_images/images/
        if not any(images_dir.glob("*.jpg")):
            for rel in ("test_images/images", "test_images", "images"):
                alt = extract_root / rel
                if alt.is_dir() and any(alt.glob("*.jpg")):
                    images_dir = alt
                    break
    return images_dir


INPUT_DIR = resolve_input_dir()
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
IMAGES_DIR = setup_images_dir(INPUT_DIR, WORK_DIR)

TEST_CSV = INPUT_DIR / "test.csv"
SAMPLE_CSV = INPUT_DIR / "sample_submission.csv"
OUTPUT_CSV = WORK_DIR / "submission.csv"
CHECKPOINT_CSV = WORK_DIR / "checkpoint.csv"

test_df = pd.read_csv(TEST_CSV)
image_ids_on_disk = {p.stem for p in IMAGES_DIR.glob("*.jpg")}

train_labels_df = None
for labels_path in [INPUT_DIR / "train_labels.csv", Path("public_dataset/train_labels.csv")]:
    if labels_path.exists():
        train_labels_df = pd.read_csv(labels_path, keep_default_na=False)
        break

print(f"Input dir   : {INPUT_DIR}")
print(f"Test set    : {len(test_df):,} images")
print(f"Images dir  : {IMAGES_DIR} ({len(image_ids_on_disk):,} jpg)")
print(f"Working dir : {WORK_DIR}")

if train_labels_df is not None:
    ocr_fill = (train_labels_df["ocr_text"].astype(str).str.strip() != "").mean()
    prod_fill = (train_labels_df["product_name"].astype(str).str.strip() != "").mean()
    print(f"Train labels: {len(train_labels_df):,} rows (draft v1 | OCR {ocr_fill:.1%} | product {prod_fill:.1%})")
    print("  Use train.csv + train_images.zip with train_labels_df for supervised training.")
else:
    print("Train labels: not found (test-only mode)")

missing = set(test_df["image_id"]) - image_ids_on_disk
if missing:
    print(f"Warning: {len(missing)} image_ids have no jpg (OCR will return empty for those)")
else:
    print("All test image_ids have matching jpg files")

print("\nPreview test.csv:")
test_df.head(3)


Input dir   : /kaggle/input/competitions/the-2nd-ura-hackathon
Test set    : 2,006 images
Images dir  : /kaggle/input/competitions/the-2nd-ura-hackathon/test_images/images (2,006 jpg)
Working dir : /kaggle/working
Train labels: 4,892 rows (draft v1 | OCR 79.8% | product 59.3%)
  Use train.csv + train_images.zip with train_labels_df for supervised training.
All test image_ids have matching jpg files

Preview test.csv:


,image_id
0,img_2934
1,img_2935
2,img_2936


## Cell 3 — Brand Rules (Zero-Parameter Baseline)

Regex dictionary — **no training, no model file**, runs in microseconds.

Use this pattern when you need deterministic, deployable product extraction before adding any ML head.


In [3]:
# BRAND_RULES: (regex, canonical_name, [product_line_keywords])
BRAND_RULES = [
    # PATE / HA LONG (dominant in test set)
    (r"ha\s*long\s*canfoco.*pate.*c[ộo]t|c[ộo]t\s*đ[èe]n.*ha\s*long\s*canfoco", "Ha Long Canfoco Pate Cột Đèn", []),
    (r"ha\s*long\s*canfoco.*pate|canfoco.*pate\s*c[ộo]t|pate\s*c[ộo]t\s*đ[èe]n.*canfoco", "Ha Long Canfoco Pate", []),
    (r"ha\s*long\s*canfoco|halong\s*canfoco|canfood|canfoco", "Ha Long Canfoco", []),
    (r"đ[ồo]\s*h[ộo]p\s*h[ạa]\s*long|do\s*hop\s*ha\s*long", "Đồ Hộp Hạ Long", []),
    (r"pate\s*c[ộo]t\s*đ[èe]n|pate\s*cot\s*den|c[ộo]t\s*đ[èe]n\s*h[ảa]i\s*ph[òo]ng", "Pate Cột Đèn Hải Phòng", []),
    (r"h[ạa]\s*long\s*pate|pate\s*h[ạa]\s*long", "Ha Long Canfoco Pate", []),
    # MILK / DAIRY
    (r"vinamilk", "Vinamilk", ["flex", "adm gold", "sure", "canxi",
                                 "colosbaby", "colos baby", "ong tho", "ông thọ", "dielac", "grow"]),
    (r"th true|thtrue", "TH True Milk", ["true yogurt", "grow", "school milk", "true butter"]),
    (r"dutch lady|cô gái", "Dutch Lady", ["grow", "complete", "canxi", "mom"]),
    (r"nutifood|nuti\b", "Nutifood", ["growplus", "grow plus", "pedia", "iq"]),
    (r"ensure\b", "Abbott Ensure", ["gold", "original", "plus"]),
    (r"pediasure", "Abbott PediaSure", []),
    (r"similac", "Abbott Similac", []),
    (r"glucerna", "Abbott Glucerna", []),
    (r"milo\b", "Nestlé Milo", []),
    (r"nestle|nestlé", "Nestlé", ["milo", "coffee mate", "carnation", "nestea", "nan", "sữa bột"]),
    (r"aptamil", "Aptamil", []),
    (r"friso\b", "Friso", ["gold", "comfort", "prestige"]),
    (r"meiji\b", "Meiji", ["growing", "step"]),
    (r"ba vi\b|bavi\b|ba vì", "Ba Vì", ["gold"]),
    (r"lothamilk", "Lothamilk", ["canxi"]),
    (r"yomost", "Yomost", []),
    (r"dalat milk|đà lạt", "Đà Lạt Milk", []),
    (r"kun\b|kun milk", "Kun", ["chocolate", "strawberry"]),
    (r"fami\b", "Fami", ["canxi", "kid"]),
    (r"anlene", "Anlene", ["gold", "concentrate"]),
    (r"anchor\b", "Anchor", ["butter", "cream"]),
    # PATE / CANNED MEAT (other brands)
    (r"vissan", "Vissan", ["pate heo", "pate ga", "pate gà",
                           "xuc xich", "xúc xích", "lap xuong", "lạp xưởng"]),
    (r"hafi\b", "Hafi", ["pate"]),
    (r"ba huan|ba huân", "Ba Huân", ["pate"]),
    (r"san ha\b|san hà", "San Hà", ["pate"]),
    (r"\bcp\b|c\.p\.", "CP", ["pate", "xúc xích"]),
    (r"long bien|long biên", "Long Biên", ["pate"]),
    (r"\bpate\b|patê", "Pate", []),
    # ADD YOUR OWN BELOW
    # (r"regex", "Brand", ["line1", "line2"]),
]

def extract_product(text: str) -> str:
    """Return 'Brand ProductLine', 'Brand', or '' if no match."""
    if not text or not text.strip():
        return ""
    tl = text.lower().replace("patê", "pate")
    for pattern, brand, lines in BRAND_RULES:
        if re.search(pattern, tl, re.IGNORECASE):
            for line in lines:
                if re.search(line, tl, re.IGNORECASE):
                    return f"{brand} {line.replace('+', '+').title()}"
            return brand
    return ""

tests = [
    ("HA LONG CANFOCO Pate Cột Đèn tạm dừng sản xuất", "Ha Long Canfoco Pate Cột Đèn"),
    ("HALONG CANFOCO pate cot den Hải Phòng", "Ha Long Canfoco Pate Cột Đèn"),
    ("ĐỒ HỘP HẠ LONG ISO 22000", "Đồ Hộp Hạ Long"),
    ("Pate cột đèn ai khóc nỗi đau này", "Pate Cột Đèn Hải Phòng"),
    ("Sữa tươi Vinamilk Flex 180ml giảm 20%", "Vinamilk Flex"),
    ("MILO Nestle chocolate 3 in 1", "Nestlé Milo"),
    ("Pate Heo Vissan 170g combo 3 hộp", "Vissan Pate Heo"),
    ("TH True Milk tươi tiệt trùng ít béo", "TH True Milk"),
    ("Dutch Lady Grow+ 900g", "Dutch Lady Grow"),
    ("No brand in this text", ""),
]
all_pass = True
for text, expected in tests:
    got = extract_product(text)
    ok = got == expected
    all_pass = all_pass and ok
    status = "PASS" if ok else "FAIL"
    print(f"{status}: '{text[:45]}' -> got='{got}' | expected='{expected}'")

print()
print("All self-tests passed." if all_pass else "Some self-tests failed — check BRAND_RULES.")
print(f"Total rules loaded: {len(BRAND_RULES)}")


PASS: 'HA LONG CANFOCO Pate Cột Đèn tạm dừng sản xuấ' -> got='Ha Long Canfoco Pate Cột Đèn' | expected='Ha Long Canfoco Pate Cột Đèn'
PASS: 'HALONG CANFOCO pate cot den Hải Phòng' -> got='Ha Long Canfoco Pate Cột Đèn' | expected='Ha Long Canfoco Pate Cột Đèn'
PASS: 'ĐỒ HỘP HẠ LONG ISO 22000' -> got='Đồ Hộp Hạ Long' | expected='Đồ Hộp Hạ Long'
PASS: 'Pate cột đèn ai khóc nỗi đau này' -> got='Pate Cột Đèn Hải Phòng' | expected='Pate Cột Đèn Hải Phòng'
PASS: 'Sữa tươi Vinamilk Flex 180ml giảm 20%' -> got='Vinamilk Flex' | expected='Vinamilk Flex'
PASS: 'MILO Nestle chocolate 3 in 1' -> got='Nestlé Milo' | expected='Nestlé Milo'
PASS: 'Pate Heo Vissan 170g combo 3 hộp' -> got='Vissan Pate Heo' | expected='Vissan Pate Heo'
PASS: 'TH True Milk tươi tiệt trùng ít béo' -> got='TH True Milk' | expected='TH True Milk'
PASS: 'Dutch Lady Grow+ 900g' -> got='Dutch Lady Grow' | expected='Dutch Lady Grow'
PASS: 'No brand in this text' -> got='' | expected=''

All self-tests passed.
Total rules loaded

## Cell 3b — Train Lightweight Product Head

Trains in **seconds on CPU**. Serialized size typically **2–5 MB**.

| Component | Size | Inference |
|---|---|---|
| `BRAND_RULES` | 0 MB | ~0.01 ms |
| Sklearn head (this cell) | ~few MB | ~0.5 ms |
| EasyOCR (Cell 4) | ~200 MB | ~1–2 s/img |

**Pattern:** rules first → binary "has product?" gate → TF-IDF char n-grams + LogisticRegression.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


class ProductPredictor:
        def __init__(self, min_class_count=3, prob_threshold=0.60, max_features=3000):
            self.min_class_count = min_class_count
            self.prob_threshold = prob_threshold
            self.max_features = max_features
            self._has_clf = self._prod_clf = None
            self._n_train = self._n_classes = 0

        def fit(self, train_labels, rule_fn):
            df = train_labels.copy()
            df["ocr_text"] = df["ocr_text"].astype(str).str.strip()
            df["product_name"] = df["product_name"].astype(str).str.strip()
            self._rule_fn = rule_fn
            self._has_clf = Pipeline([
                ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=self.max_features, min_df=2)),
                ("clf", LogisticRegression(max_iter=400, class_weight="balanced")),
            ])
            self._has_clf.fit(df["ocr_text"], (df["product_name"] != "").astype(int))
            pos = df[(df["ocr_text"] != "") & (df["product_name"] != "")]
            keep = pos["product_name"].value_counts()
            keep = keep[keep >= self.min_class_count].index
            pos = pos[pos["product_name"].isin(keep)]
            self._prod_clf = Pipeline([
                ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=self.max_features, min_df=2)),
                ("clf", LogisticRegression(max_iter=400, class_weight="balanced")),
            ])
            if len(pos):
                self._prod_clf.fit(pos["ocr_text"], pos["product_name"])
            self._n_train, self._n_classes = len(df), pos["product_name"].nunique() if len(pos) else 0
            return self

        def predict(self, ocr_text):
            ocr_text = "" if ocr_text is None else str(ocr_text).strip()
            if not ocr_text:
                return ""
            ruled = self._rule_fn(ocr_text)
            if ruled:
                return ruled
            if self._has_clf is None or self._prod_clf is None:
                return ""
            proba = self._has_clf.predict_proba([ocr_text])[0]
            if 1 not in self._has_clf.classes_ or proba[list(self._has_clf.classes_).index(1)] < self.prob_threshold:
                return ""
            return str(self._prod_clf.predict([ocr_text])[0])


product_predictor = None
if train_labels_df is not None:
    product_predictor = ProductPredictor(min_class_count=3, prob_threshold=0.60, max_features=3000)
    product_predictor.fit(train_labels_df, extract_product)
    pos = train_labels_df.copy()
    pos["ocr_text"] = pos["ocr_text"].astype(str).str.strip()
    pos["product_name"] = pos["product_name"].astype(str).str.strip()
    n_pairs = ((pos["ocr_text"] != "") & (pos["product_name"] != "")).sum()
    print(f"Trained lightweight product head on {len(train_labels_df):,} rows ({n_pairs:,} OCR+product pairs)")
    if hasattr(product_predictor, "summary"):
        print(product_predictor.summary())
    if hasattr(product_predictor, "model_size_mb"):
        import time
        t0 = time.perf_counter()
        for _ in range(500):
            product_predictor.predict("Vinamilk Flex 180ml Ha Long Canfoco")
        ms = (time.perf_counter() - t0) / 500 * 1000
        print(f"  Product inference: ~{ms:.2f} ms/image (CPU, after OCR)")
else:
    print("train_labels.csv not found — rules-only mode (simplest lightweight path)")


def predict_product(ocr_text: str) -> str:
    if product_predictor is not None:
        return product_predictor.predict(ocr_text)
    return extract_product(ocr_text)

Trained lightweight product head on 4,892 rows (2,871 OCR+product pairs)


## Cell 4 — OCR Engine (Demo Backend)

EasyOCR is **not lightweight** (~200 MB) — included so the notebook runs end-to-end on Kaggle without API keys.

**For deployment:** replace `run_ocr()` internals with a smaller OCR and keep `predict_product()` unchanged.

In [5]:
import easyocr
import numpy as np

reader = easyocr.Reader(["vi", "en"], gpu=False, verbose=False)
print("EasyOCR loaded (Vietnamese + English, CPU mode)")


def load_image(image_id: str):
    """Load image from images/ (extracted from images.zip in Cell 2)."""
    path = IMAGES_DIR / f"{image_id}.jpg"
    if not path.exists():
        return None
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None


def preprocess(img, max_dim: int = 1280):
    """Resize, sharpen lightly, and boost contrast for OCR."""
    w, h = img.size
    if max(w, h) > max_dim:
        ratio = max_dim / max(w, h)
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    img = ImageEnhance.Contrast(img).enhance(1.35)
    return img.filter(ImageFilter.SHARPEN)


def postprocess_ocr(text: str) -> str:
    """Normalize whitespace and drop consecutive duplicate tokens (common in thumbnails)."""
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    if not tokens:
        return ""
    deduped = [tokens[0]]
    for tok in tokens[1:]:
        if tok.lower() != deduped[-1].lower():
            deduped.append(tok)
    return " ".join(deduped)


def run_ocr(image_id: str) -> tuple:
    """Return (ocr_text, product_name)."""
    img = load_image(image_id)
    if img is None:
        return "", ""

    img = preprocess(img)
    img_np = np.array(img)

    try:
        results = reader.readtext(img_np, detail=1, paragraph=False)
        results = sorted(results, key=lambda r: (r[0][0][1], r[0][0][0]))
        lines = [r[1] for r in results if r[2] > 0.35]
        ocr_text = postprocess_ocr(" ".join(lines))
    except Exception:
        ocr_text = ""

    return ocr_text, predict_product(ocr_text)


print("\nSmoke test on first image...")
first_id = test_df["image_id"].iloc[0]
ocr, prod = run_ocr(first_id)
prod_rules = extract_product(ocr)
print(f"  image_id    : {first_id}")
print(f"  ocr_text    : {ocr[:80]}{'...' if len(ocr) > 80 else ''}")
print(f"  product     : {prod or '(empty)'}")
if prod != prod_rules:
    print(f"  rules-only  : {prod_rules or '(empty)'}  (train model override)")


EasyOCR loaded (Vietnamese + English, CPU mode)

Smoke test on first image...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  image_id    : img_2934
  ocr_text    : HALONG CANFUCO j TIkTok @hanoinewsIheannzb HÀ NỘl NEWS
  product     : (empty)


## Cell 5 — Main Loop

Processes test images on **CPU**. Most time is OCR (~1–2 s/image); product head adds <1 ms.

Expected composite score: **~0.5** (reference starter — improve accuracy while keeping models small for Final Presentation).

In [6]:
SAVE_EVERY = 50

done_ids = set()
results = []

if CHECKPOINT_CSV.exists():
    ckpt = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    done_ids = set(ckpt["image_id"])
    results = ckpt.to_dict("records")
    print(f"Resuming from checkpoint: {len(done_ids):,} images done")
else:
    print("Starting fresh")

pending = [i for i in test_df["image_id"] if i not in done_ids]
print(f"Pending: {len(pending):,} | Done: {len(done_ids):,}")
print()

errors = 0
for idx, img_id in enumerate(tqdm(pending, desc="OCR Progress")):
    ocr_text, product_name = run_ocr(img_id)

    if ocr_text == "" and (IMAGES_DIR / f"{img_id}.jpg").exists():
        errors += 1

    results.append({
        "image_id": img_id,
        "ocr_text": ocr_text,
        "product_name": product_name,
    })

    if (idx + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")

pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")

df_result = pd.DataFrame(results)
ocr_fill = (df_result["ocr_text"].str.strip() != "").mean()
prod_fill = (df_result["product_name"].str.strip() != "").mean()

print()
print(f"Processed     : {len(df_result):,}")
print(f"OCR fill rate : {ocr_fill:.1%}")
print(f"Product fill  : {prod_fill:.1%}")
print(f"OCR failures  : {errors:,}")


Starting fresh
Pending: 2,006 | Done: 0



OCR Progress:   0%|          | 0/2006 [00:00<?, ?it/s]


Processed     : 2,006
OCR fill rate : 79.0%
Product fill  : 44.3%
OCR failures  : 421


## Cell 6 — Validate and Export submission.csv

Validate submission format before uploading to Kaggle. All checks must pass.

In [7]:
import csv
import sys

sys.path.insert(0, str(Path(".").resolve()))
try:
    from build_dataset import write_submission_csv
except ImportError:
    def write_submission_csv(df, path):
        out = df[["image_id", "ocr_text", "product_name"]].copy()
        for col in ("ocr_text", "product_name"):
            out[col] = out[col].fillna("").astype(str).str.strip()
            out.loc[out[col] == "", col] = " "
        out.to_csv(path, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

sub = pd.DataFrame(results)[["image_id", "ocr_text", "product_name"]]
sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)

print("Validating submission format...\n")
checks = {}

expected_ids = set(sample["image_id"])
got_ids = set(sub["image_id"])
checks["AC-1 Row count match"] = len(sub) == len(sample)
checks["AC-2 No extra IDs"] = len(got_ids - expected_ids) == 0
checks["AC-3 No missing IDs"] = len(expected_ids - got_ids) == 0
checks["AC-4 No duplicate IDs"] = not sub["image_id"].duplicated().any()
checks["AC-5 No null values"] = not sub.isnull().any().any()
checks["AC-6 No newline in text"] = not sub["ocr_text"].str.contains(r"\n|\t", regex=True, na=False).any()
checks["AC-7 Columns correct"] = list(sub.columns) == ["image_id", "ocr_text", "product_name"]

all_pass = True
for name, ok in checks.items():
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {name}")
    if not ok:
        all_pass = False

print()
if all_pass:
    sub = sub.set_index("image_id").reindex(sample["image_id"]).reset_index()
    sub["ocr_text"] = sub["ocr_text"].fillna("").astype(str).str.strip()
    sub["product_name"] = sub["product_name"].fillna("").astype(str).str.strip()
    write_submission_csv(sub, OUTPUT_CSV)

    print("All checks passed.")
    print(f"Saved to: {OUTPUT_CSV}")
    print("\nFirst 5 rows:")
    display(sub.head())
    ocr_fill = (sub["ocr_text"].str.strip() != "").mean()
    prod_fill = (sub["product_name"].str.strip() != "").mean()
    print(f"\nStats: OCR fill={ocr_fill:.1%} | Product fill={prod_fill:.1%} | Rows={len(sub):,}")
    print("\nNext steps:")
    print("  1. Kaggle -> Competition -> Submit Predictions")
    print("  2. Upload submission.csv from /kaggle/working/")
    print("  3. Check score on the Public Leaderboard")
else:
    print("Validation failed — fix issues above before submitting.")


Validating submission format...

  [PASS] AC-1 Row count match
  [PASS] AC-2 No extra IDs
  [PASS] AC-3 No missing IDs
  [PASS] AC-4 No duplicate IDs
  [PASS] AC-5 No null values
  [PASS] AC-6 No newline in text
  [PASS] AC-7 Columns correct

All checks passed.
Saved to: /kaggle/working/submission.csv

First 5 rows:


,image_id,ocr_text,product_name
0,img_2934,HALONG CANFUCO j TIkTok @hanoinewsIheannzb HÀ ...,
1,img_2935,,
2,img_2936,Op News 11/01/2026 1 9 6 7 N C E 5 / 0 ĐO TAM ...,
3,img_2937,,
4,img_2938,Op News 11/01/2026 1 9 6 7 c E N 9 CANFOCO HAL...,Ha Long Canfoco



Stats: OCR fill=79.0% | Product fill=44.3% | Rows=2,006

Next steps:
  1. Kaggle -> Competition -> Submit Predictions
  2. Upload submission.csv from /kaggle/working/
  3. Check score on the Public Leaderboard


## Cell 7 — Local Evaluation (Composite Score)

Score predictions with a **standalone inline** metric (same formula as Kaggle).

**Local dev:** uses `smce_dataset/solution.csv` (BTC only — never upload to participants).  
**Compare:** rules-only vs train-enhanced product model on the same OCR output.

In [8]:
import json


def _inline_composite_score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """Fallback metric identical to the competition formula."""

    def _clean(val) -> str:
        return "" if pd.isna(val) else str(val).strip()

    required_cols = {"ocr_text", "product_name"}
    if row_id_column_name not in solution.columns or row_id_column_name not in submission.columns:
        raise ValueError(f"Missing id column: {row_id_column_name}")
    if not required_cols.issubset(solution.columns) or not required_cols.issubset(submission.columns):
        raise ValueError("Both solution and submission must contain ocr_text and product_name")

    sub_ids = submission[row_id_column_name]
    sol_ids = solution[row_id_column_name]
    if sub_ids.duplicated().any():
        raise ValueError("Duplicate image_id in submission")
    if set(sub_ids) != set(sol_ids):
        missing = len(set(sol_ids) - set(sub_ids))
        extra = len(set(sub_ids) - set(sol_ids))
        raise ValueError(f"Submission IDs must match solution exactly (missing {missing}, extra {extra})")

    merged = solution.merge(submission, on=row_id_column_name, suffixes=("_gt", "_pred"), how="inner")
    if merged.empty:
        raise ValueError("No matching rows after merge")

    def token_f1(gt, pred) -> float:
        gt = _clean(gt)
        pred = _clean(pred)
        if not gt and not pred:
            return 1.0
        gt_tokens = set(gt.lower().split())
        pred_tokens = set(pred.lower().split())
        if not gt_tokens or not pred_tokens:
            return 0.0
        common = gt_tokens & pred_tokens
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(gt_tokens)
        if precision + recall == 0:
            return 0.0
        return 2 * precision * recall / (precision + recall)

    def cer(gt, pred) -> float:
        gt = _clean(gt)
        pred = _clean(pred)
        if len(gt) == 0:
            return 0.0 if len(pred) == 0 else 1.0
        m, n = len(gt), len(pred)
        dp = list(range(n + 1))
        for i in range(1, m + 1):
            prev, dp[0] = dp[0], i
            for j in range(1, n + 1):
                temp = dp[j]
                dp[j] = prev if gt[i - 1] == pred[j - 1] else 1 + min(prev, dp[j], dp[j - 1])
                prev = temp
        return min(dp[n] / len(gt), 1.0)

    product_f1 = merged.apply(lambda r: token_f1(r["product_name_gt"], r["product_name_pred"]), axis=1).mean()
    avg_cer = merged.apply(lambda r: cer(r["ocr_text_gt"], r["ocr_text_pred"]), axis=1).mean()
    return round(float(0.6 * product_f1 + 0.4 * (1.0 - avg_cer)), 4)


# Standalone metric: do not load any external notebook/python file.
score_fn = _inline_composite_score
print("Using standalone inline composite score")

solution_paths = [
    Path("smce_dataset/solution.csv"),
    Path("/kaggle/input/smce-solution/solution.csv"),
]
solution_df = None
for p in solution_paths:
    if p.exists():
        solution_df = pd.read_csv(p, keep_default_na=False)
        print(f"Loaded solution: {p} ({len(solution_df):,} rows)")
        break

if solution_df is None:
    print("solution.csv not found — skip local scoring (OK on Kaggle participant side)")
elif not results:
    print("No OCR predictions yet — running oracle ablation on GT ocr_text (product head only)\n")
    gt = solution_df[["image_id", "ocr_text", "product_name"]]
    for c in ["ocr_text", "product_name"]:
        gt[c] = gt[c].astype(str).str.strip()
    sub_rules = gt.copy()
    sub_rules["product_name"] = sub_rules["ocr_text"].map(extract_product)
    sub_train = gt.copy()
    sub_train["product_name"] = sub_train["ocr_text"].map(predict_product)
    print(f"  Rules-only product:     {score_fn(gt, sub_rules, 'image_id'):.4f}")
    print(f"  Train-enhanced product: {score_fn(gt, sub_train, 'image_id'):.4f}")
    print("\nRun Cell 5 for full pipeline score (OCR + product model)")
else:
    pred_df = pd.DataFrame(results)[["image_id", "ocr_text", "product_name"]]
    pred_rules = pred_df.copy()
    pred_rules["product_name"] = pred_rules["ocr_text"].map(extract_product)

    gt = solution_df[["image_id", "ocr_text", "product_name"]]
    s_full = score_fn(gt, pred_df, "image_id")
    s_rules = score_fn(gt, pred_rules, "image_id")

    print("\nComposite score (full pipeline — OCR + train product model):")
    print(f"  Score: {s_full:.4f}")

    print("\nAblation (same OCR, rules-only product):")
    print(f"  Score: {s_rules:.4f}")

    if solution_df is not None and "Usage" in solution_df.columns:
        for split in ["Public", "Private"]:
            ids = set(solution_df.loc[solution_df["Usage"] == split, "image_id"])
            if not ids:
                continue
            gt_s = gt[gt["image_id"].isin(ids)]
            pred_s = pred_df[pred_df["image_id"].isin(ids)]
            print(f"  {split}: {score_fn(gt_s, pred_s, 'image_id'):.4f}")

    sample_empty = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
    sample_empty["ocr_text"] = ""
    sample_empty["product_name"] = ""
    print(f"\nReference — empty sample_submission: {score_fn(gt, sample_empty, 'image_id'):.4f}")

Using standalone inline composite score
solution.csv not found — skip local scoring (OK on Kaggle participant side)


## Cell 8 — Next Steps (Beat ~0.5, Stay Lightweight)

This notebook targets **~0.5** as a **reference**, not SOTA. Round 3 judges **CPU efficiency, model size, inference speed**.

| Priority | Idea | Lightweight? |
|---|---|---|
| 1 | Swap EasyOCR → PaddleOCR mobile / Tesseract | Yes |
| 2 | Expand `BRAND_RULES` | Yes (0 MB) |
| 3 | Tune `max_features` / `prob_threshold` in Cell 3b | Yes (few MB) |
| 4 | Quantized or distilled vision model | Medium |

**Avoid for Final Presentation:** large vision LLMs, GPU-only pipelines, multi-GB checkpoints.

SMCE Challenge 2026 | RAISE Lab, HCMUT
